# 07 — Proporção de CPFs por Taxa de Leitura e Entrega

**Objetivo:** dar suporte à **Parte 3 — Desenho de Experimento (Teste A/B)**, fornecendo estatísticas de baseline a nível de CPF que podem ser utilizadas como métricas primárias/secundárias e para o cálculo de tamanho de amostra.

Especificamente, este notebook calcula:

1. **Proporção de CPFs com taxa de leitura ≥ 90%** — taxa de leitura = `read / total_disparos` por CPF.
2. **Proporção de CPFs com taxa de entrega ≥ 90%** — taxa de entrega = `(delivered + read) / total_disparos` por CPF (mensagem entregue inclui as que foram lidas).

**Fonte:** `outputs/processed/base_disparo.parquet` (gerado em `01_preprocessing.ipynb`).

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PARQUET_PATH = PROJECT_ROOT / 'outputs' / 'processed' / 'base_disparo.parquet'

df = pd.read_parquet(PARQUET_PATH)
print(f'Linhas: {len(df):,} | CPFs únicos: {df["cpf"].nunique():,}')
df['status_disparo'].value_counts()

## Agregação por CPF

- `total_disparos`: número total de disparos do CPF.
- `n_read`: disparos com `status_disparo == 'read'`.
- `n_delivered_or_read`: disparos com status `delivered` ou `read` (entrega efetiva — `read` implica que a mensagem foi entregue antes de ser lida).
- `taxa_leitura = n_read / total_disparos`
- `taxa_entrega = n_delivered_or_read / total_disparos`

In [ ]:
df['is_read'] = (df['status_disparo'] == 'read').astype(int)
df['is_delivered_or_read'] = df['status_disparo'].isin(['delivered', 'read']).astype(int)

agg_cpf = (
    df.groupby('cpf')
      .agg(total_disparos=('status_disparo', 'size'),
           n_read=('is_read', 'sum'),
           n_delivered_or_read=('is_delivered_or_read', 'sum'))
      .reset_index()
)
agg_cpf['taxa_leitura'] = agg_cpf['n_read'] / agg_cpf['total_disparos']
agg_cpf['taxa_entrega'] = agg_cpf['n_delivered_or_read'] / agg_cpf['total_disparos']
agg_cpf.head()

In [ ]:
agg_cpf[['total_disparos', 'taxa_leitura', 'taxa_entrega']].describe()

## Proporção de CPFs com taxa ≥ 90%

In [ ]:
THRESHOLD = 0.90
n_cpfs = len(agg_cpf)

n_read_90 = (agg_cpf['taxa_leitura'] >= THRESHOLD).sum()
n_entrega_90 = (agg_cpf['taxa_entrega'] >= THRESHOLD).sum()

prop_read_90 = n_read_90 / n_cpfs
prop_entrega_90 = n_entrega_90 / n_cpfs

resultado = pd.DataFrame({
    'metrica': ['taxa_leitura >= 90%', 'taxa_entrega (delivered+read) >= 90%'],
    'n_cpfs_atende': [n_read_90, n_entrega_90],
    'n_cpfs_total': [n_cpfs, n_cpfs],
    'proporcao': [prop_read_90, prop_entrega_90],
})
resultado['proporcao_pct'] = (resultado['proporcao'] * 100).round(2).astype(str) + '%'
resultado

In [ ]:
print(f'Total de CPFs analisados: {n_cpfs:,}')
print(f'CPFs com taxa de leitura >= 90%:  {n_read_90:,} ({prop_read_90:.2%})')
print(f'CPFs com taxa de entrega >= 90%:  {n_entrega_90:,} ({prop_entrega_90:.2%})')

## Uso na Parte 3 (Teste A/B)

- **Baseline da métrica primária** (taxa de leitura por CPF) e da **métrica secundária** (taxa de entrega por CPF) para dimensionar o experimento.
- A proporção de CPFs com taxa ≥ 90% serve como referência para o efeito mínimo detectável e para definir o público elegível ao teste.